In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [2]:
device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Using device:", device)

Using device: mps


In [3]:
train_data = np.load("../data/processed/train_sequences.npz")
val_data = np.load("../data/processed/validation_sequences.npz")
test_data = np.load("../data/processed/test_sequences.npz")

X_train = train_data["X"]
y_train = train_data["y"]

X_val = val_data["X"]
y_val = val_data["y"]

X_test = test_data["X"]
y_test = test_data["y"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (23973, 12, 207)
y_train: (23973, 207)
X_val: (3410, 12, 207)
y_val: (3410, 207)
X_test: (6838, 12, 207)
y_test: (6838, 207)


In [4]:
adjacency_matrix = np.load(
    "../data/processed/adjacency_matrix.npy"
)

with open("../data/processed/graph_sensor_ids.txt", "r") as f:
    graph_sensor_ids = [
        sensor_id.strip()
        for sensor_id in f.read().split(",")
        if sensor_id.strip()
    ]

print("Adjacency matrix:", adjacency_matrix.shape)
print("Number of sensor IDs:", len(graph_sensor_ids))

print("First 10 sensor IDs:")
print(graph_sensor_ids[:10])

Adjacency matrix: (207, 207)
Number of sensor IDs: 207
First 10 sensor IDs:
['773869', '767541', '767542', '717447', '717446', '717445', '773062', '767620', '737529', '717816']


In [5]:
assert X_train.shape[2] == len(graph_sensor_ids)
assert adjacency_matrix.shape == (
    len(graph_sensor_ids),
    len(graph_sensor_ids)
)

print("Traffic sensors:", X_train.shape[2])
print("Graph sensors:", len(graph_sensor_ids))
print("Graph shape:", adjacency_matrix.shape)

print("\nSensor ordering verified.")

Traffic sensors: 207
Graph sensors: 207
Graph shape: (207, 207)

Sensor ordering verified.


In [6]:

X_train_gnn = X_train[:, -1, :]
X_val_gnn = X_val[:, -1, :]
X_test_gnn = X_test[:, -1, :]

print("X_train_gnn:", X_train_gnn.shape)
print("X_val_gnn:", X_val_gnn.shape)
print("X_test_gnn:", X_test_gnn.shape)
print("y_train:", y_train.shape)

X_train_gnn: (23973, 207)
X_val_gnn: (3410, 207)
X_test_gnn: (6838, 207)
y_train: (23973, 207)


In [7]:

A = torch.tensor(
    adjacency_matrix,
    dtype=torch.float32,
    device=device
)

print("Adjacency tensor shape:", A.shape)
print("Device:", A.device)
print("Data type:", A.dtype)

Adjacency tensor shape: torch.Size([207, 207])
Device: mps:0
Data type: torch.float32


In [8]:

I = torch.eye(
    A.size(0),
    device=device,
    dtype=torch.float32
)

A_with_self_loops = A + I

degree = A_with_self_loops.sum(dim=1)

degree_inv_sqrt = torch.pow(
    degree,
    -0.5
)

A_normalized = (
    degree_inv_sqrt.unsqueeze(1)
    * A_with_self_loops
    * degree_inv_sqrt.unsqueeze(0)
)

print("Normalized adjacency shape:", A_normalized.shape)
print("Contains NaN:", torch.isnan(A_normalized).any().item())
print("Contains Inf:", torch.isinf(A_normalized).any().item())

Normalized adjacency shape: torch.Size([207, 207])
Contains NaN: False
Contains Inf: False


In [9]:
class GraphConvolution(nn.Module):

    def __init__(self, in_features, out_features):
        super().__init__()

        self.linear = nn.Linear(
            in_features,
            out_features
        )

    def forward(self, x, adjacency):

        aggregated = torch.matmul(
            adjacency,
            x
        )

        output = self.linear(
            aggregated
        )

        return output

In [11]:
class TrafficGNN(nn.Module):

    def __init__(
        self,
        num_nodes,
        hidden_size=64,
        output_size=1
    ):
        super().__init__()

        self.gcn1 = GraphConvolution(
            1,
            hidden_size
        )

        self.gcn2 = GraphConvolution(
            hidden_size,
            hidden_size
        )

        self.output_layer = nn.Linear(
            hidden_size,
            output_size
        )

    def forward(self, x, adjacency):

        x = x.unsqueeze(-1)

        x = self.gcn1(
            x,
            adjacency
        )

        x = torch.relu(x)

        x = self.gcn2(
            x,
            adjacency
        )

        x = torch.relu(x)

        x = self.output_layer(x)

        return x.squeeze(-1)

In [12]:
gnn_model = TrafficGNN(
    num_nodes=207,
    hidden_size=64,
    output_size=1
).to(device)

print(gnn_model)

TrafficGNN(
  (gcn1): GraphConvolution(
    (linear): Linear(in_features=1, out_features=64, bias=True)
  )
  (gcn2): GraphConvolution(
    (linear): Linear(in_features=64, out_features=64, bias=True)
  )
  (output_layer): Linear(in_features=64, out_features=1, bias=True)
)


In [13]:
sample_input = torch.tensor(
    X_train_gnn[:8],
    dtype=torch.float32,
    device=device
)

sample_output = gnn_model(
    sample_input,
    A_normalized
)

print("Input shape:", sample_input.shape)
print("Output shape:", sample_output.shape)

Input shape: torch.Size([8, 207])
Output shape: torch.Size([8, 207])


In [14]:

A_normalized = A.clone()

degree = A_normalized.sum(dim=1)

degree_inv_sqrt = torch.pow(
    degree,
    -0.5
)

A_normalized = (
    degree_inv_sqrt.unsqueeze(1)
    * A_normalized
    * degree_inv_sqrt.unsqueeze(0)
)

print("Normalized adjacency shape:", A_normalized.shape)
print("Contains NaN:", torch.isnan(A_normalized).any().item())
print("Contains Inf:", torch.isinf(A_normalized).any().item())

Normalized adjacency shape: torch.Size([207, 207])
Contains NaN: False
Contains Inf: False


In [15]:
BATCH_SIZE = 64

train_dataset = TensorDataset(
    torch.tensor(X_train_gnn, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32)
)

val_dataset = TensorDataset(
    torch.tensor(X_val_gnn, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.float32)
)

test_dataset = TensorDataset(
    torch.tensor(X_test_gnn, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.float32)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Training batches: 375
Validation batches: 54
Test batches: 107


In [16]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    gnn_model.parameters(),
    lr=0.001
)

print("Loss:", criterion)
print("Optimizer:", optimizer)

Loss: MSELoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [17]:
batch_x, batch_y = next(iter(train_loader))

batch_x = batch_x.to(device)
batch_y = batch_y.to(device)

gnn_model.train()

optimizer.zero_grad()

predictions = gnn_model(
    batch_x,
    A_normalized
)

loss = criterion(
    predictions,
    batch_y
)

loss.backward()

optimizer.step()

print("Input:", batch_x.shape)
print("Target:", batch_y.shape)
print("Prediction:", predictions.shape)
print("Loss:", loss.item())

Input: torch.Size([64, 207])
Target: torch.Size([64, 207])
Prediction: torch.Size([64, 207])
Loss: 0.7865469455718994


In [18]:

train_target_mask = np.load(
    "../data/processed/train_target_mask.npy"
)

val_target_mask = np.load(
    "../data/processed/validation_target_mask.npy"
)

test_target_mask = np.load(
    "../data/processed/test_target_mask.npy"
)

print("Train target mask:", train_target_mask.shape)
print("Validation target mask:", val_target_mask.shape)
print("Test target mask:", test_target_mask.shape)

print("\nTrain observed targets:",
      train_target_mask.sum())

print("Validation observed targets:",
      val_target_mask.sum())

print("Test observed targets:",
      test_target_mask.sum())

Train target mask: (23973, 207)
Validation target mask: (3410, 207)
Test target mask: (6838, 207)

Train observed targets: 4665451
Validation observed targets: 664844
Test observed targets: 1266610


In [19]:

train_dataset = TensorDataset(
    torch.tensor(X_train_gnn, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32),
    torch.tensor(train_target_mask, dtype=torch.bool)
)

val_dataset = TensorDataset(
    torch.tensor(X_val_gnn, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.float32),
    torch.tensor(val_target_mask, dtype=torch.bool)
)

test_dataset = TensorDataset(
    torch.tensor(X_test_gnn, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.float32),
    torch.tensor(test_target_mask, dtype=torch.bool)
)

BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Training batches: 375
Validation batches: 54
Test batches: 107


In [20]:
def masked_mse_loss(predictions, targets, mask):
    
    mask = mask.float()
    
    squared_error = (predictions - targets) ** 2
    
    masked_error = squared_error * mask
    
    return masked_error.sum() / mask.sum().clamp_min(1.0)

In [21]:

gnn_model = TrafficGNN(
    num_nodes=207,
    hidden_size=64,
    output_size=1
).to(device)

optimizer = torch.optim.Adam(
    gnn_model.parameters(),
    lr=0.001
)

print("GNN reinitialized.")

GNN reinitialized.


In [25]:

import random

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Random seed:", SEED)

Random seed: 42


In [26]:
EPOCHS = 20
PATIENCE = 4

best_val_loss = float("inf")
patience_counter = 0

train_history = []
val_history = []

best_model_path = "../models/gnn_best.pth"

for epoch in range(EPOCHS):

    gnn_model.train()

    total_train_loss = 0.0
    train_batches = 0

    for batch_x, batch_y, batch_mask in train_loader:

        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        batch_mask = batch_mask.to(device)

        optimizer.zero_grad()

        predictions = gnn_model(
            batch_x,
            A_normalized
        )

        loss = masked_mse_loss(
            predictions,
            batch_y,
            batch_mask
        )

        loss.backward()

        optimizer.step()

        total_train_loss += loss.item()
        train_batches += 1

    average_train_loss = (
        total_train_loss / train_batches
    )

    gnn_model.eval()

    total_val_loss = 0.0
    val_batches = 0

    with torch.no_grad():

        for batch_x, batch_y, batch_mask in val_loader:

            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            batch_mask = batch_mask.to(device)

            predictions = gnn_model(
                batch_x,
                A_normalized
            )

            loss = masked_mse_loss(
                predictions,
                batch_y,
                batch_mask
            )

            total_val_loss += loss.item()
            val_batches += 1

    average_val_loss = (
        total_val_loss / val_batches
    )

    train_history.append(average_train_loss)
    val_history.append(average_val_loss)

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"Train Loss: {average_train_loss:.6f} | "
        f"Val Loss: {average_val_loss:.6f}"
    )

    if average_val_loss < best_val_loss:

        best_val_loss = average_val_loss
        patience_counter = 0

        torch.save(
            gnn_model.state_dict(),
            best_model_path
        )

        print("  → Best GNN model saved.")

    else:

        patience_counter += 1

        if patience_counter >= PATIENCE:

            print("  → Early stopping triggered.")
            break

Epoch 01/20 | Train Loss: 0.673141 | Val Loss: 0.653348
  → Best GNN model saved.
Epoch 02/20 | Train Loss: 0.672746 | Val Loss: 0.658545
Epoch 03/20 | Train Loss: 0.672646 | Val Loss: 0.654429
Epoch 04/20 | Train Loss: 0.672758 | Val Loss: 0.652883
  → Best GNN model saved.
Epoch 05/20 | Train Loss: 0.671964 | Val Loss: 0.653249
Epoch 06/20 | Train Loss: 0.672390 | Val Loss: 0.651849
  → Best GNN model saved.
Epoch 07/20 | Train Loss: 0.671787 | Val Loss: 0.653255
Epoch 08/20 | Train Loss: 0.672442 | Val Loss: 0.651821
  → Best GNN model saved.
Epoch 09/20 | Train Loss: 0.671510 | Val Loss: 0.651105
  → Best GNN model saved.
Epoch 10/20 | Train Loss: 0.671388 | Val Loss: 0.651321
Epoch 11/20 | Train Loss: 0.671648 | Val Loss: 0.651682
Epoch 12/20 | Train Loss: 0.671584 | Val Loss: 0.651997
Epoch 13/20 | Train Loss: 0.671255 | Val Loss: 0.655981
  → Early stopping triggered.


In [27]:

gnn_model.load_state_dict(
    torch.load(
        "../models/gnn_best.pth",
        map_location=device
    )
)

gnn_model.eval()

print("Best GNN model restored.")
print("Best validation loss:", best_val_loss)

Best GNN model restored.
Best validation loss: 0.6511051354032976


In [28]:

gnn_val_predictions = []

with torch.no_grad():

    for batch_x, batch_y, batch_mask in val_loader:

        batch_x = batch_x.to(device)

        predictions = gnn_model(
            batch_x,
            A_normalized
        )

        gnn_val_predictions.append(
            predictions.cpu().numpy()
        )

gnn_val_predictions = np.concatenate(
    gnn_val_predictions,
    axis=0
)

print("GNN validation predictions:",
      gnn_val_predictions.shape)

print("Validation targets:",
      y_val.shape)

GNN validation predictions: (3410, 207)
Validation targets: (3410, 207)


In [29]:
import joblib

scaler = joblib.load(
    "../data/processed/traffic_scaler.pkl"
)

gnn_val_predictions_original = scaler.inverse_transform(
    gnn_val_predictions
)

y_val_original = scaler.inverse_transform(
    y_val
)

print(
    "Prediction range:",
    gnn_val_predictions_original.min(),
    "to",
    gnn_val_predictions_original.max(),
    "mph"
)

print(
    "Target range:",
    y_val_original.min(),
    "to",
    y_val_original.max(),
    "mph"
)

Prediction range: -23.92421 to 74.509224 mph
Target range: 0.4999962 to 70.0 mph


In [30]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

pred_flat = gnn_val_predictions_original.ravel()
target_flat = y_val_original.ravel()

mask_flat = val_target_mask.ravel()

pred_observed = pred_flat[mask_flat]
target_observed = target_flat[mask_flat]

gnn_val_mae = mean_absolute_error(
    target_observed,
    pred_observed
)

gnn_val_rmse = np.sqrt(
    mean_squared_error(
        target_observed,
        pred_observed
    )
)

print("GNN Validation MAE :", gnn_val_mae)
print("GNN Validation RMSE:", gnn_val_rmse)

GNN Validation MAE : 4.715280055999756
GNN Validation RMSE: 7.856982737309215


In [31]:

negative_predictions = np.sum(
    gnn_val_predictions_original < 0
)

above_max_predictions = np.sum(
    gnn_val_predictions_original > 70
)

total_predictions = gnn_val_predictions_original.size

print("Total predictions:", total_predictions)

print(
    "Negative predictions:",
    negative_predictions
)

print(
    "Negative prediction percentage:",
    100 * negative_predictions / total_predictions
)

print(
    "Predictions above 70 mph:",
    above_max_predictions
)

print(
    "Above-70 prediction percentage:",
    100 * above_max_predictions / total_predictions
)

Total predictions: 705870
Negative predictions: 628
Negative prediction percentage: 0.08896822361057985
Predictions above 70 mph: 7709
Above-70 prediction percentage: 1.0921274455636307


In [32]:
gnn_results = pd.DataFrame({
    "model": ["GNN"],
    "validation_mae": [gnn_val_mae],
    "validation_rmse": [gnn_val_rmse]
})

gnn_results.to_csv(
    "../results/gnn_validation_results.csv",
    index=False
)

print(gnn_results)
print("\nSaved to ../results/gnn_validation_results.csv")

  model  validation_mae  validation_rmse
0   GNN         4.71528         7.856983

Saved to ../results/gnn_validation_results.csv


In [33]:

gnn_training_history = pd.DataFrame({
    "epoch": range(1, len(train_history) + 1),
    "train_loss": train_history,
    "validation_loss": val_history
})

gnn_training_history.to_csv(
    "../results/gnn_training_history.csv",
    index=False
)

print(gnn_training_history)

print(
    "\nSaved to ../results/gnn_training_history.csv"
)

    epoch  train_loss  validation_loss
0       1    0.673141         0.653348
1       2    0.672746         0.658545
2       3    0.672646         0.654429
3       4    0.672758         0.652883
4       5    0.671964         0.653249
5       6    0.672390         0.651849
6       7    0.671787         0.653255
7       8    0.672442         0.651821
8       9    0.671510         0.651105
9      10    0.671388         0.651321
10     11    0.671648         0.651682
11     12    0.671584         0.651997
12     13    0.671255         0.655981

Saved to ../results/gnn_training_history.csv
